# HW2｜邊界規則

**個人作業　W5 出題 → W7 繳交**

---

## 這次作業要做什麼

HW1 你只做檢索：輸入問題，輸出條文片段，到此為止。這次把**生成**接上去。

一接上去，馬上會冒出一個 HW1 沒有的問題：

> **模型永遠會給你一個答案，就算資料裡根本沒有。**

這不是猜的。HW1 的第 10 題（碩士班修業年限）在語料裡查不到，但檢索器回傳的
最高相似度是 **0.88**——而查得到的題目是 0.88 到 0.92。**兩者幾乎一樣。**

所以你的第一個直覺——設一個相似度門檻，低於它就拒答——**做不到**。
分數看不出「我不知道」。

**這次作業的核心不是「怎麼答得更好」，是「怎麼知道什麼時候不該答」。**

---

## 你要做出什麼

一個規章問答助理，會做四件事：

| 它會 | 舉例 |
|---|---|
| 回答，並附上出處條號 | 「碩士班畢業至少 36 學分（第六條(四)）」 |
| **說「本規則未規範」** | 問修業年限——這批語料裡真的沒有 |
| **發現規定不適用時說清楚** | 問產業實作課程——那是大學部的規定 |
| **反問你哪一年入學** | 問英語授課課程門數——108 到 111 學年沒這條，112 學年起要修 1 門 |

最後那一項是這次作業最有意思的地方，而且**不是我們硬加的對話功能**。

系上的修業規則有六個年度版本。「碩士班要修幾門英語授課專業課程？」這個問題，
對 108–111 學年入學的人來說**規則裡根本沒有這一條**，對 112 學年起入學的人是**要修 1 門**。

**同一個問題，兩個答案，取決於你哪一年入學。助理不問，就答不出來。**

---

## 這份 notebook 怎麼用

**看到 ✍️ 開頭的格子，就是你要作答的地方。** 雙擊那一格開始打字，把表格的空欄位填滿。

**標有 `TODO` 的程式碼格要你補完，這次共四處。** 跟 HW1 不同，這次不是只改一行，
而是要寫完整的函式或判斷區塊——每一格都標了要做什麼、預期輸出長什麼樣。
沒改就執行，會跳出中文訊息告訴你卡在哪。

**其他程式碼格直接執行就好**，不用改。**檢索的部分整套都給你了**，
就是 HW1 的完成版——這次不用再處理切塊跟排名。

---

## 三件明確不做的事

| | 為什麼 |
|---|---|
| **不讓模型決定要查幾次** | 檢索次數由你的程式寫死（最多重搜一次）。讓模型自己決定會導致每次跑的結果都不一樣，你就沒辦法診斷了 |
| **不做多輪對話** | 只有一種第二輪：發現版本分歧時反問入學年度。不用記住三輪以前說過什麼 |
| **不加校級語料、不接網路搜尋** | 語料就是系級＋院級這 22 份。校級要 HW3／HW4 才開放，提前用會讓「規定未載」這一型消失 |

---

## 評分重點

| 項目 | 配分 |
|---|---|
| 可運作 | 25 |
| 邊界與安全 | 25 |
| **失敗診斷** | **30** |
| 成本意識 | 20 |

**失敗診斷永遠最重。** 「你的助理在哪裡壞掉、你憑什麼判斷、你改了什麼」——
這是最難請人代做的部分，也是這門課真正要教的。

還有一件事先講明：**把所有題目都拒答，不會拿到高分。**
誤傷正常題跟該拒沒拒一樣嚴重，而且 5-1 要你自己指出這件事。

---
# 區塊一　假設設定

1-3 編號在後，但**時間上最先完成**。猜測必須在你動手之前寫下，否則區塊五就沒有對照的基準。

### 1-3　事前猜測　→　請寫在**作答本第 1 題**

### 1-1　名詞定義　→　請寫在**作答本第 2 題**

### 1-2　流程說明　→　請寫在**作答本第 3 題**

---
# 環境設定

執行階段請選 **T4 GPU**（執行階段 → 變更執行階段類型 → 硬體加速器 → T4 GPU）。

底下四格直接執行就好。

### ⚠ 裝完套件要重新啟動工作階段

下一格會升級幾個 Colab 預載的套件。**升級之後必須重新啟動**，
否則記憶體裡還是舊版本，等一下連 Vertex AI 時會出現這個錯誤：

```
AttributeError: module 'google.auth.credentials' has no attribute
                'CredentialsWithRegionalAccessBoundary'
```

**做法**：跑完下一格 → 執行階段 → **重新啟動工作階段**（或 `Ctrl+M` 再按 `.`）
→ 回到最上面重新執行。第二次跑 pip 那格會很快，因為套件已經裝好了。

只用 `local` 後端的話不會遇到這個問題，但重啟一次也無妨。

In [ ]:
# 安裝套件（約 1 分鐘）
!pip -q install sentence-transformers transformers accelerate
!pip -q install -U google-genai google-auth    # 用 Gemini／Vertex AI 才需要

# ⚠ 跑完這一格請「重新啟動工作階段」，再從頭執行一次。

# 預期輸出：幾行安裝訊息。**最後會有兩行紅色的 pip 相依性警告**
#   （google-colab 要求 google-auth==2.49.0，而這裡裝的是較新版）——
#   那是正常的，不影響後面任何一格，不需要處理。

In [ ]:
# 下載語料
!git clone -q https://github.com/eddieli002/fju-im-agent-hw.git /content/repo
!cp -r /content/repo/hw-corpus /content/hw-corpus

import glob
print("系級：", len(glob.glob("/content/hw-corpus/text/01-系級/*.txt")), "份")
print("院級：", len(glob.glob("/content/hw-corpus/02-院級/*.md")), "份")

# 預期輸出：系級 19 份　院級 3 份

## 選擇語言模型後端

**這次要跑三次，用兩個後端：**

| 跑次 | 後端 | 內容 |
|---|---|---|
| **A** | `local`（Qwen2.5-3B，跑在這台 Colab 上） | 有評估器 |
| **B** | 你自己選 | 有評估器 |
| **C** | 同 B | **關掉評估器** |

**A 是全班共同的基準線**，大家用同一個模型、同一組參數，數據才能互相比較。

**B 你可以選任何後端**——學校給的 GCP 額度、你自己的 API key、
或是免費方案。**如果你沒有任何 API 可用，B 就直接用 `local`，
那 A 和 B 是同一次跑，你只要跑兩次就好。** 這樣不會少學到任何東西。

### 四個可選的後端

| 代號 | 是什麼 | 認證方式 | 吃誰的額度 |
|---|---|---|---|
| `local` | Qwen2.5-3B，跑在這台 Colab 上 | 不需要 | 無（免費） |
| `aistudio` | Gemini API，從 aistudio.google.com 拿金鑰 | 金鑰 | 免費層 |
| `vertex` | 在 Google Cloud 專案裡呼叫 Gemini | **Google 帳號授權，不用金鑰** | GCP 額度 |
| `openai_compatible` | Groq、OpenRouter 等 | 金鑰 | 各平台自訂 |

> **⚠ 憑證絕對不要直接貼在程式碼裡。**
>
> 用 Colab 左側的鑰匙圖示（🔑）新增密鑰，**名稱**填 `GOOGLE_API_KEY`、值貼你的金鑰，
> 並把「筆記本存取權」打開。程式裡寫的是 `userdata.get("GOOGLE_API_KEY")`
> ——括號裡放的是**名稱**，不是金鑰本身。
>
> 你會把這份 notebook 交出來。金鑰貼進去就是外洩，而且交出去之後收不回來。
>
> `vertex` 這個選項不需要金鑰（改用 Google 帳號授權），是四個裡面最不容易出事的。

In [ ]:
# ── 後端設定：改這裡來切換 ────────────────────────────────
BACKEND = "local"    # "local" / "aistudio" / "vertex" / "openai_compatible"

# 只有 BACKEND = "vertex" 時要填這兩個
GCP_PROJECT = "你的-gcp-專案-id"       # Cloud Console 首頁就看得到
GCP_LOCATION = "us-central1"
# ──────────────────────────────────────────────────────────

import time, re, json, unicodedata
import torch

# 呼叫紀錄：每一次呼叫模型都會記在這裡，區塊四的成本統計靠它
CALLS = []
MAX_CALLS = 200          # 預算斷路器：超過就停，避免無限迴圈燒掉額度


class Budget(Exception):
    pass


def _record(kind, n_in, n_out, sec):
    if len(CALLS) >= MAX_CALLS:
        raise Budget(f"已達呼叫上限 {MAX_CALLS} 次。若確定需要更多，請調高 MAX_CALLS。")
    CALLS.append({"kind": kind, "in": n_in, "out": n_out, "sec": sec})


if BACKEND == "local":
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"      # 共同基準線，不要改
    _tok = AutoTokenizer.from_pretrained(MODEL_ID)
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto", torch_dtype=torch.float16)
    _model.eval()

    def llm(prompt, kind="其他", max_new_tokens=256):
        msgs = [{"role": "user", "content": prompt}]
        text = _tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = _tok(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            # do_sample=False → 每次跑結果完全一樣。
            # 這是本機模型相對線上 API 的最大優勢：可重現的失敗才診斷得動。
            out = _model.generate(**enc, max_new_tokens=max_new_tokens,
                                  do_sample=False, pad_token_id=_tok.eos_token_id)
        sec = time.time() - t0
        ans = _tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        _record(kind, enc["input_ids"].shape[1], len(out[0]) - enc["input_ids"].shape[1], sec)
        return ans.strip()

elif BACKEND in ("aistudio", "vertex"):
    # 兩者用同一個 SDK（google-genai），只有建立 Client 的方式不同。
    # 注意：舊的 google.generativeai 套件已停止維護，不要再用。
    from google import genai
    # ⚠ 型號代號會變動。2026-09-01 實測時 gemini-2.5-flash 已對新使用者關閉，
    #   錯誤訊息會直接告訴你該換成哪一個。要查目前可用的型號：
    #       for m in _client.models.list(): print(m.name)
    MODEL_ID = "gemini-3.6-flash"              # 換成你實際用的型號

    if BACKEND == "aistudio":
        from google.colab import userdata
        _key = userdata.get("GOOGLE_API_KEY")  # ← 這裡放的是「名稱」，不是金鑰本身
        if not _key or len(_key) < 20:
            raise ValueError(
                "找不到金鑰。請在 Colab 左側 🔑 面板新增一個名為 GOOGLE_API_KEY 的密鑰，"
                "把金鑰貼在「值」欄位，並打開「筆記本存取權」。"
                "程式裡的 userdata.get() 括號內只放名稱。")
        _client = genai.Client(api_key=_key)
    else:
        # Vertex AI：用 Google 帳號授權，不需要金鑰
        from google.colab import auth
        auth.authenticate_user()               # 跳出授權視窗，選有額度的帳號
        _client = genai.Client(vertexai=True, project=GCP_PROJECT,
                               location=GCP_LOCATION)

    def llm(prompt, kind="其他", max_new_tokens=256):
        t0 = time.time()
        r = _client.models.generate_content(model=MODEL_ID, contents=prompt)
        sec = time.time() - t0
        u = getattr(r, "usage_metadata", None)
        _record(kind,
                (getattr(u, "prompt_token_count", 0) or 0) if u else 0,
                (getattr(u, "candidates_token_count", 0) or 0) if u else 0,
                sec)
        return (r.text or "").strip()

else:
    # Groq、OpenRouter 等 OpenAI 相容介面
    from google.colab import userdata
    from openai import OpenAI
    MODEL_ID = "llama-3.3-70b-versatile"       # 換成你實際用的型號
    _key = userdata.get("LLM_API_KEY")
    if not _key or len(_key) < 20:
        raise ValueError(
            "找不到金鑰。請在 Colab 左側 🔑 面板新增一個名為 LLM_API_KEY 的密鑰。")
    _c = OpenAI(api_key=_key, base_url="https://api.groq.com/openai/v1")

    def llm(prompt, kind="其他", max_new_tokens=256):
        t0 = time.time()
        r = _c.chat.completions.create(
            model=MODEL_ID, messages=[{"role": "user", "content": prompt}],
            max_tokens=max_new_tokens, temperature=0)
        sec = time.time() - t0
        _record(kind, r.usage.prompt_tokens, r.usage.completion_tokens, sec)
        return r.choices[0].message.content.strip()

print("後端：", BACKEND, "／", MODEL_ID)

# 預期輸出（local）：下載進度條，最後印出「後端： local ／ Qwen/Qwen2.5-3B-Instruct」
#   首次下載約 5 分鐘、佔 7.1 GB 顯示記憶體
# 預期輸出（vertex）：跳出 Google 帳號授權視窗，授權後印出「後端： vertex ／ gemini-2.5-flash」
#   若出現 403，表示 Cloud Console 尚未啟用 Vertex AI API（aiplatform.googleapis.com）

In [ ]:
# 確認後端通了。
# ⚠ 金鑰「存在」不等於「有效」——上一格只檢查了長度，過期的金鑰照樣通過。
#   真正會撞到的是這一格，所以在這裡把錯誤翻成看得懂的話。
try:
    print(llm("請只回答兩個字：測試", kind="連線測試"))
    print("呼叫紀錄：", CALLS[-1])
except Exception as e:
    print("連線測試失敗：", type(e).__name__)
    print(" ", str(e).split(chr(10))[0][:160])
    print()
    print("常見原因：")
    print("  1. 金鑰過期或被撤銷 → 重新申請一把，更新 Colab 左側 🔑 面板裡的值")
    print("  2. 型號代號變了 → 見上一格 MODEL_ID 的註解，用 models.list() 查目前可用的")
    print("  3. 免費額度用完 → 換一個後端（BACKEND 改成 local 就完全不需要金鑰）")
    raise SystemExit("後端沒通，先解決上面的問題再往下跑。")

# 預期輸出：模型回覆（內容不重要，有回東西就好）＋ 一筆呼叫紀錄

---
# 區塊二　素材整備

這次新增**院級語料**（管理學院的三份規定）。它們跟系級規章有重疊，
但適用對象不一定相同——那正是這次要處理的問題之一。

底下兩格是 HW1 的完成版，直接執行。

In [ ]:
# 載入語料並切塊（HW1 完成版，不用改）
import glob, os, re

def load_docs():
    docs = []
    for p in sorted(glob.glob("/content/hw-corpus/text/01-系級/*.txt")):
        docs.append({"file": os.path.basename(p), "level": "系級",
                     "text": open(p, encoding="utf-8").read()})
    for p in sorted(glob.glob("/content/hw-corpus/02-院級/*.md")):
        docs.append({"file": os.path.basename(p), "level": "院級",
                     "text": open(p, encoding="utf-8").read()})
    return docs


def chunk_fixed(docs, size=500, overlap=100):
    """每 size 字一塊，相鄰重疊 overlap 字。與 HW1 完全相同。"""
    chunks, step = [], size - overlap
    for d in docs:
        t, n = d["text"], 0
        for i in range(0, len(t), step):
            piece = t[i:i + size]
            if piece.strip():
                chunks.append({"file": d["file"], "level": d["level"],
                               "seq": n, "text": piece})
                n += 1
    return chunks


DOCS = load_docs()
CHUNKS = chunk_fixed(DOCS)
print(f"文件 {len(DOCS)} 份 → 塊 {len(CHUNKS)}"
      f"（系級 {sum(1 for c in CHUNKS if c['level']=='系級')}"
      f"、院級 {sum(1 for c in CHUNKS if c['level']=='院級')}）")

# 預期輸出：文件 22 份 → 塊 357（系級 339、院級 18）

In [ ]:
# 建索引與檢索（HW1 完成版，不用改）
from sentence_transformers import SentenceTransformer
import numpy as np

_emb_model = SentenceTransformer("intfloat/multilingual-e5-small")
EMB = np.array(_emb_model.encode(["passage: " + c["text"] for c in CHUNKS],
                                 normalize_embeddings=True, batch_size=64,
                                 show_progress_bar=True))


def search(query, k=5):
    q = _emb_model.encode("query: " + query, normalize_embeddings=True)
    scores = EMB @ q
    top = np.argsort(-scores)[:k]
    return [{"rank": r + 1, "score": float(scores[i]), **CHUNKS[i]}
            for r, i in enumerate(top)]


def show(results):
    for r in results:
        print(f"#{r['rank']} {r['score']:.4f} [{r['level']}] {r['file'][:30]}#{r['seq']}")
        print("   ", r["text"][:90].replace("\n", " "), "…")


show(search("碩士班畢業至少須修滿多少學分？"))

# 預期輸出：5 筆結果，分數在 0.90 上下，前幾筆來自系修業規則

### 2-1　院級語料檢視　→　請寫在**作答本第 4 題**

In [ ]:
# 觀察工具（不用改）：印出某個問題撈回來的片段，並標出關鍵字有沒有出現
def inspect(question, marks):
    """marks 是你想檢查的字串清單，例如 ["大學部", "碩士班"]。"""
    print("問題：", question, "\n")
    for r in search(question):
        hit = [m for m in marks if m in r["text"]]
        print(f"#{r['rank']} {r['score']:.4f} [{r['level']}] {r['file'][:28]}#{r['seq']}")
        print(f"     含有：{hit if hit else '（都沒有）'}")


inspect("碩士班學生的英文檢定畢業規定是什麼？", ["大學部", "碩士班"])

# 預期輸出：5 筆結果，每筆列出它含有哪些關鍵字

### 2-2　兩種誤判的觀察

用上面的 `inspect()` 工具，找出這兩種狀況各一個實例。可以自己換問題、換要檢查的字串。

### 一、誤判未載

**狀況**：答案其實在語料裡，但助理撈不到，於是說「本規則未規範」。

常見的兩種成因：

- **用詞對不上**——你問「指導教授什麼時候要找好」，規章寫的是「碩士論文指導教授申請書」
- **答案被切散了**——條文寫著「另行公告」「詳見附錄」，真正的答案在另一塊

### 二、適用範圍脫落

**狀況**：撈到了，但那一條規定管的不是發問的人，助理卻照著回答。

常見的兩種成因：

- **限定對象寫在標題**——檔名寫「大學部」，但你問的是碩士班
- **切塊把限定對象切掉了**——標題和條文分開，助理看到的那一塊沒有「大學部」三個字

> **請把答案寫在作答本第 5 題**，不要寫在這一格。

### 2-3　自己出四題

**這是這次作業最重要的一格。** 你要自己找出四個會讓助理出錯的題目。

| 類別 | 題數 | 限制 |
|---|---|---|
| **規定未載** | 2 | 至少 1 題必須是「**規定存在，但不適用於發問的人**」，不可以兩題都是「文件裡真的沒寫」 |
| **規定分歧** | 2 | 1 題是**不同檔案給不同答案**、1 題是**同一段文字裡就有好幾代規定**（例如一條規定後面接著「自 112 學年度起改為⋯」，新舊兩版擠在同一塊裡）|

**「憑什麼認定」那一欄不能只寫「我查過了」。** 要寫出你查了哪幾份文件、
哪幾條，以及你看到什麼才下這個判斷。這一欄就是這一格的重點。

> **一個實用的檢查方法**：判斷一題「有沒有分歧」，不要數撈回幾個不同的檔案，
> 要數**撈回幾個互相矛盾的答案**。五個不同年度的檔案，可能全部都不含你要的那個數字。

> **請把答案寫在作答本第 6 題**，不要寫在這一格。

---
# 區塊三　判準確立

下面 12 題就是你要逐題送進助理的問題。全班統一，所以數據可以互相比較。

> **關於第 8 題**：它問「大學部要修滿多少專業必修學分」，語料裡確實有答案
> （不同年度是 67 或 64），照理說助理該反問你哪一年入學。
>
> 但這一題的**期望行為是「拒答」**。原因是：檢索器兩次都沒把含有 67／64 的片段
> 撈回來，助理手上根本沒有那個數字。**在它看得到的證據面前，拒答是負責任的行為。**
>
> 這一題要教的是：**助理做對了，系統整體還是給不出答案——因為問題出在檢索。**
> 那正是 HW1 在處理的事。

In [ ]:
# 評測集（不用改）
#
#   期望行為   助理最後應該做什麼
#   期望標籤   評估器「應該」判成哪一類——這是教師的判斷，你不一定要同意（見 3-1）
#   答案值     這一題在語料裡的候選答案。**區塊四的重搜對照要用它**——
#              判斷重搜有沒有帶來新資訊，看的是「有沒有出現新的答案值」，
#              不是「有沒有撈到新片段」。拒答題沒有數值答案，此欄為空。
#   使用者補充 助理反問時，測試腳本會自動餵這句話進去

EVAL = [
 {"id":1,"型":"A 正常","q":"碩士班畢業至少須修滿多少學分？",
  "期望行為":"回答","期望標籤":"正確","答案值":["36 學分"]},
 {"id":2,"型":"A 正常","q":"碩士班前二學年每學期選課的上限與下限各為多少學分？",
  "期望行為":"回答","期望標籤":"正確","答案值":["選課上限為 18 學分"]},
 {"id":3,"型":"B 真未載","q":"碩士班修業年限最長幾年？",
  "期望行為":"拒答","期望標籤":"錯誤","答案值":[]},
 {"id":4,"型":"B 真未載","q":"碩士班學生在什麼情況下會被退學？",
  "期望行為":"拒答","期望標籤":"錯誤","答案值":[]},
 {"id":5,"型":"C 適用對象不符","q":"碩士班學生可以修產業實作課程嗎？",
  "期望行為":"拒答","期望標籤":"錯誤","答案值":[]},
 {"id":6,"型":"C 適用對象不符","q":"碩士生沒通過英檢，需要參加管理學院辦的 8 次英語自學方案測驗嗎？",
  "期望行為":"拒答","期望標籤":"錯誤","答案值":[]},
 {"id":7,"型":"D 多版本","q":"碩士班學生畢業前需要修幾門以英語授課的專業課程？",
  "期望行為":"反問","期望標籤":"模稜兩可","答案值":["1 門"],
  "使用者補充":"我是 110 學年度入學"},
 # 第 8 題的期望行為是「拒答」而不是「反問」——原因見下方說明。
 {"id":8,"型":"D 多版本","q":"大學部要修滿多少專業必修學分？",
  "期望行為":"拒答","期望標籤":"錯誤",
  "答案值":["專業必修課程 67 學分","專業必修課程 64 學分"],
  "使用者補充":"我是 109 學年度入學"},
 {"id":9,"型":"D 多版本","q":"全人教育核心課程要修幾學分？",
  "期望行為":"反問","期望標籤":"模稜兩可",
  "答案值":["全人教育核心課程 8 學分","全人教育核心課程 10 學分"],
  "使用者補充":"我是 114 學年度入學"},
 {"id":10,"型":"E 同段多代","q":"大學部要修幾門本院開設的英語授課專業課程？",
  "期望行為":"反問","期望標籤":"模稜兩可",
  "答案值":["5門（或15學分）","3學分","選修一門"],
  "使用者補充":"我是 113 學年度入學"},
 {"id":11,"型":"F 指向他處","q":"管理學院規定的 CEFR B2 高階級對應到 TOEIC 幾分？",
  "期望行為":"回答","期望標籤":"模稜兩可","答案值":["750","785"]},
 {"id":12,"型":"F 指向他處","q":"管理學院承認哪些英語能力檢測項目？",
  "期望行為":"回答","期望標籤":"模稜兩可","答案值":["OPT","IELTS"]},
]

for e in EVAL:
    v = f"　答案值 {e['答案值']}" if e["答案值"] else ""
    print(f"{e['id']:>2}. [{e['型']}] {e['q']}{v}")

# 預期輸出：12 行題目，其中 8 題附有答案值

### 3-1　判準

**第一問：怎麼確認助理附的條號沒有唬你？**

助理回答時會像這樣附上出處：

> 碩士班畢業至少須修滿 30 學分。（依據：第六條）

它有可能**答案對但條號錯**，也有可能**條號根本是編的**。
如果現在要你用人工檢查這一句，你會怎麼做？把步驟寫下來。

（回家你會把這套步驟寫成程式，那就是 TODO 4。）

**第二問：怎麼知道一題「規定有分歧」？**

**分歧**＝ 同一個問題，語料裡有**兩個以上互相衝突的答案**。
例如一份寫 TOEIC 要 750 分、另一份寫 785 分——那就是分歧。

現在假設你撈回五塊，要判斷有沒有分歧：

**第三問：評測集的「期望標籤」，你都同意嗎？**

往上捲到「評測集」那一格，每一題都有一個 `期望標籤` 欄位——
那是**我認為**評估器應該判成哪一類（正確／錯誤／模稜兩可）。

那是我的判斷，不是標準答案。有幾題的界線本來就模糊，例如：
**撈到的規定管的是別人，這算「錯誤」還是「模稜兩可」？**

逐題看過之後，不同意的寫在下面。
**都同意也可以**，但要寫出你是看過哪幾題才這樣說。

> **請把答案寫在作答本第 7 題**，不要寫在這一格。

### 3-2　失敗預測　→　請寫在**作答本第 8 題**

---
# 區塊四　實作驗證

## 助理的流程

```
使用者問題
  └─▶ 檢索（已經給你了，k=5）
        └─▶ 評估器：判三類                        ◀── TODO 1：提示詞
              │
              ├─ 正確 ──────────────────▶ 生成 ＋ 附條號
              │
              ├─ 錯誤 ────┐                        ◀── TODO 2：分流
              │           ├─▶ 改寫查詢，重搜一次    ◀── TODO 3：改寫
              └─ 模稜兩可 ┘         │
                                    └─▶ 再判一次（到此為止，不再重搜）
                                          ├─ 還是「錯誤」   ─▶ 拒答
                                          ├─ 補齊了        ─▶ 生成
                                          ├─ 有互相矛盾的答案 ─▶ 反問入學年度
                                          └─ 沒有新東西     ─▶ 回答並標註不確定
                    ↓
              引用查核：回答裡的條號有沒有出現在證據裡   ◀── TODO 4
```

**注意「錯誤」也會重搜。** 你可能覺得判成「錯誤」就該直接拒答——
但「這些片段裡沒有答案」不等於「整個資料庫裡沒有答案」，
換個問法再查一次可能就找到了。找過兩次還是沒有，再拒答才負責任。

**重搜上限是 1 次。** 第二次判完之後，不管結果如何都不再查了。
這個上限是寫死的，不由模型決定——這樣你的每一次執行才會得到一樣的結果。

In [ ]:
# ══════════════════ TODO 1 ══════════════════
# 寫出評估器的提示詞。
#
# 它要讓模型看完「問題 ＋ 撈回來的片段」之後，只回答三個標籤中的一個：
#     正確 / 錯誤 / 模稜兩可
#
# 提示詞裡必須有 {question} 和 {chunks} 兩個位置，程式會把實際內容填進去。
# 三個標籤各代表什麼，你要自己想清楚並寫進去——寫得越明確，模型判得越穩。
# ═══════════════════════════════════════════

EVAL_PROMPT = None      # ← 換成你的提示詞字串

# 預期輸出：這一格不印東西。沒填的話，下一格會告訴你。

In [ ]:
# 評估器（不用改）
LABELS = {"模稜兩可": ["模稜兩可", "模棱两可"],
          "錯誤": ["錯誤", "错误"],
          "正確": ["正確", "正确"]}


def parse_label(text):
    """從模型輸出裡抓標籤。簡體也接受——模型偶爾會吐簡體字。"""
    for lab, alias in LABELS.items():
        if any(a in text for a in alias):
            return lab
    return None


def evaluate(question, chunks):
    if EVAL_PROMPT is None:
        # TODO 1 還沒寫。這裡不呼叫模型，直接回「正確」，
        # 讓你先看看「助理對每一題都自信作答」是什麼樣子——
        # 那個畫面就是這次作業要解決的問題。
        print("  ⚠ EVAL_PROMPT 還沒填，暫時把每一題都當成「正確」")
        return "正確"
    block = "\n\n".join(f"[片段 {c['rank']}｜{c['file']}]\n{c['text']}" for c in chunks)
    raw = llm(EVAL_PROMPT.format(question=question, chunks=block),
              kind="評估", max_new_tokens=16)
    return parse_label(raw) or "錯誤"      # 解析不出來就當作證據不足


print(evaluate(EVAL[0]["q"], search(EVAL[0]["q"])))

# 預期輸出：一個標籤（正確／錯誤／模稜兩可），或是 TODO 1 未完成的警告

In [ ]:
# ══════════════════ TODO 2 ══════════════════
# 寫出分流邏輯：拿到標籤之後，下一步該做什麼？
#
# 回傳值只能是這四個字串之一：
#     "生成"  "重搜"  "拒答"  "反問"
#
# 三個輸入：
#     label        評估器給的標籤
#     second_pass  這是不是第二次評估（True 表示已經重搜過了）
#     has_conflict 片段裡有沒有互相矛盾的答案（下一格會算好給你）
#
# 對照上面的流程圖來寫。特別注意「錯誤」在第一次和第二次的處理不一樣。
# ═══════════════════════════════════════════

def route(label, second_pass, has_conflict):
    result = None      # ← 換成你的判斷邏輯（if / elif / else）

    if result is None:
        raise NotImplementedError("請先完成 TODO 2：寫出分流邏輯")
    return result


# 快速自我檢查
print(route("正確", False, False), "← 應該是 生成")
print(route("錯誤", False, False), "← 應該是 重搜")
print(route("錯誤", True, False), "← 應該是 拒答")
print(route("模稜兩可", True, True), "← 應該是 反問")

# 預期輸出：四行，每行的結果都跟箭頭後面說的一樣

In [ ]:
# ══════════════════ TODO 3 ══════════════════
# 寫出查詢改寫：第一次沒撈到需要的東西，換個問法再查一次。
#
# 輸入：原本的問題 question，以及第一次撈回來的片段 chunks
# 輸出：一個新的查詢字串
#
# 想一下——第一次撈到的片段裡，有沒有什麼線索可以幫你組出更好的查詢？
# 例如片段裡出現「自 112 學年度入學新生起實施」這種字樣，就暗示還有別的版本。
# ═══════════════════════════════════════════

def rewrite_query(question, chunks):
    new_q = None      # ← 換成你的改寫邏輯

    if new_q is None:
        raise NotImplementedError("請先完成 TODO 3：寫出查詢改寫")
    return new_q


print(rewrite_query(EVAL[6]["q"], search(EVAL[6]["q"])))

# 預期輸出：一個跟原問題不同、但仍然在問同一件事的查詢字串

In [ ]:
# 生成與矛盾偵測（不用改）
#
# 這個提示詞刻意要求「兩行的固定格式」，而不是用自然語氣拜託模型附條號。
# 固定格式的條號產出率明顯較高——
# 小模型對格式的服從度高於對叮嚀的服從度。
#
# **但「有附條號」不等於「條號是對的」。** 那件事你要在 4-1 診斷。

GEN_PROMPT = """你是輔仁大學資訊管理學系的規章查詢助理。

請根據以下規章片段回答問題，並嚴格照下面兩行的格式輸出，不要有其他內容：

條號：（在這裡列出你依據的條號，用頓號分隔，例如 第六條、第七條）
回答：（在這裡寫三句話以內的回答）

規則：
1. 只能依據片段的內容回答，不可以自己補充。
2. 條號必須是片段裡真的出現過的，不可以自己編。
3. 若同一件事有多個不同規定，在回答裡說明各自適用哪些學年度。

【問題】
{question}

【規章片段】
{chunks}"""


def parse_answer(raw):
    """從兩行格式裡拆出條號與回答。

    模型不一定照格式。拆不出「回答：」時，就把整段當成回答——
    寧可讓後面的引用查核抓不到條號（那是真實情況），
    也不要因為解析失敗就把整題判成錯誤。
    """
    cite = ans = ""
    for line in raw.split("\n"):
        line = line.strip()
        if line.startswith("條號"):
            cite = line.split("：", 1)[-1].split(":", 1)[-1].strip()
        elif line.startswith("回答"):
            ans = line.split("：", 1)[-1].split(":", 1)[-1].strip()
    return cite, (ans or raw.strip())


def generate(question, chunks, extra=""):
    block = "\n\n".join(f"[片段 {c['rank']}｜{c['file']}]\n{c['text']}" for c in chunks)
    q = question + ("\n（補充資訊：" + extra + "）" if extra else "")
    raw = llm(GEN_PROMPT.format(question=q, chunks=block), kind="生成", max_new_tokens=256)
    cite, ans = parse_answer(raw)
    return {"raw": raw, "條號行": cite, "回答": ans}


def norm_v(s):
    """比對答案值用：壓掉空白、markdown 的粗體星號，並做 NFKC 正規化。

    院級語料是從網頁抓的 md，關鍵數字常被標成 **785**，
    不處理星號就會比對不到。

    NFKC 是為了全形／半形。語料於 2026-09-09 做過 NFKC 正規化
    （全形括號折成半形），評測集的答案值若寫成全形就比對不到——
    第 10 題的「5門（或15學分）」就踩過這個坑。與下面 normalize() 同一個理由。
    """
    return re.sub(r"[\s*_]+", "", unicodedata.normalize("NFKC", s))


def values_in(chunks, item):
    """這些片段裡看得到這一題的哪些答案值？

    **只比對評測集標註的答案值**，不自己去抓「數字＋單位」——
    那樣會把條號、年度、無關的學分數全抓進來，量到的是雜訊。
    """
    body = norm_v(" ".join(c["text"] for c in chunks))
    return {v for v in item.get("答案值", []) if norm_v(v) in body}


def find_conflict(chunks):
    """片段裡有沒有互相矛盾的答案？用「同一種單位出現多個不同數值」當作訊號。

    這是個粗略的偵測方式，不完美——5-3 會請你評估它夠不夠用。
    """
    body = " ".join(c["text"] for c in chunks)
    for unit in ["學分", "門"]:
        vals = set(re.findall(r"(\d+)\s*" + unit, body))
        if len(vals) >= 2:
            return True
    return False


print("矛盾偵測（第 7 題）：", find_conflict(search(EVAL[6]["q"])))
print("第 11 題看得到的答案值：", values_in(search(EVAL[10]["q"]), EVAL[10]))

# 預期輸出：True 或 False，以及第 11 題的答案值（應該同時看到 750 和 785）

In [ ]:
# ══════════════════ TODO 4 ══════════════════
# 寫出引用查核：回答裡提到的條號，真的出現在片段裡嗎？
#
# 輸入：cited  從回答裡抓出來的條號清單（下面的 regex 已經幫你抓好了）
#       chunks 助理當時看到的片段
# 輸出：(通過與否, 沒對上的條號清單)
#
# 兩個提示，不注意的話一定會卡住：
#   1. normalize() 會壓掉空白。片段裡的條號長成「第 六 條」，不壓掉會對不上。
#   2. 條號後面可能跟著款號，例如「第六條(四)」。但在條文裡，
#      「第六條」和「（四）」中間往往隔了幾十個字——**款號要先丟掉，只比對「第X條」**。
# ═══════════════════════════════════════════

CITE_RE = re.compile(r"第\s*[一二三四五六七八九十百]+\s*條(?:\s*[（(]\s*[一二三四五六七八九十]+\s*[)）])?")


def normalize(s):
    # NFKC：語料偶有「CJK 相容表意文字」（U+F900–U+FAFF），字形相同但比對比不中
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", s))


def check_citations(cited, chunks):
    result = None      # ← 換成你的比對邏輯，回傳 (通過與否, 沒對上的清單)

    if result is None:
        raise NotImplementedError("請先完成 TODO 4：寫出引用查核")
    return result


# 自我檢查：這個條號存在，應該通過
print(check_citations(["第六條(四)"], search("碩士班畢業至少須修滿多少學分？")))
# 這個條號不存在，應該不通過
print(check_citations(["第九十九條"], search("碩士班畢業至少須修滿多少學分？")))

# 預期輸出：第一行 (True, [])　第二行 (False, ['第九十九條'])

In [ ]:
# 主流程（不用改）——把上面四個 TODO 串起來
def run_agent(item, use_evaluator=True, verbose=False):
    q = item["q"]
    trace = {"id": item["id"], "型": item["型"], "重搜": False,
             "標籤1": None, "標籤2": None}

    chunks = search(q)
    trace["塊1"] = [c["seq"] for c in chunks]
    trace["檔1"] = [c["file"][:14] for c in chunks]

    if not use_evaluator:
        # 對照組：不判斷，直接生成
        g = generate(q, chunks)
        trace.update({"行為": "回答", "條號行": g["條號行"], "回答": g["回答"]})
    else:
        label = evaluate(q, chunks)
        trace["標籤1"] = label
        action = route(label, False, find_conflict(chunks))

        if action == "重搜":
            trace["重搜"] = True
            q2 = rewrite_query(q, chunks)
            trace["改寫查詢"] = q2
            chunks = search(q2)
            trace["塊2"] = [c["seq"] for c in chunks]
            trace["檔2"] = [c["file"][:14] for c in chunks]
            label = evaluate(q, chunks)
            trace["標籤2"] = label
            action = route(label, True, find_conflict(chunks))

        if action == "拒答":
            trace.update({"行為": "拒答", "條號行": "", "回答": "本規則未規範。"})
        elif action == "反問":
            # 助理反問，測試腳本自動餵入使用者的補充資訊
            extra = item.get("使用者補充", "")
            trace["反問"] = "請問你是哪一學年度入學的？"
            trace["使用者回覆"] = extra
            g = generate(q, chunks, extra)
            trace.update({"行為": "反問", "條號行": g["條號行"], "回答": g["回答"]})
        else:
            g = generate(q, chunks)
            trace.update({"行為": "回答", "條號行": g["條號行"], "回答": g["回答"]})

    # 條號可能出現在「條號：」那一行，也可能被模型寫進回答裡，兩處都掃
    cited = CITE_RE.findall(trace.get("條號行", "") + " " + trace["回答"])
    cited = list(dict.fromkeys(cited))              # 去重、保留順序
    ok, missing = check_citations(cited, chunks) if cited else (True, [])
    trace.update({"引用": cited, "引用查核": ok, "查無的條號": missing,
                  "有附條號": bool(cited)})

    if verbose:
        for k, v in trace.items():
            print(f"  {k}: {v}")
    return trace


_ = run_agent(EVAL[6], verbose=True)

# 預期輸出：第 7 題的完整軌跡，包含標籤、有沒有重搜、最後的行為與回答

In [ ]:
# 跑完 12 題（不用改）
import pandas as pd


def run_all(name, use_evaluator=True):
    CALLS.clear()
    rows = []
    for item in EVAL:
        t = run_agent(item, use_evaluator=use_evaluator)
        t["期望行為"] = item["期望行為"]
        t["行為正確"] = t["行為"] == item["期望行為"]
        rows.append(t)
    df = pd.DataFrame(rows)
    df.attrs["name"] = name
    df.attrs["calls"] = list(CALLS)
    n = len(df)
    print(f"[{name}] 行為正確 {df['行為正確'].sum()}/{n}"
          f"　重搜 {df['重搜'].sum() if '重搜' in df else 0} 題"
          f"　有附條號 {df['有附條號'].sum()}/{n} 題"
          f"　引用查核未過 {(~df['引用查核']).sum()} 題"
          f"　呼叫 {len(CALLS)} 次"
          f"　共 {sum(c['sec'] for c in CALLS):.0f} 秒")
    return df


RUN_A = run_all("A 有評估器")
RUN_A[["id", "型", "標籤1", "重搜", "標籤2", "行為", "期望行為", "行為正確",
       "有附條號", "引用查核"]]

# 預期輸出：一行統計 ＋ 12 列表格。
# 行為正確不會是 12/12——那是正常的，第 9 題幾乎一定會錯。
# 「有附條號」也不會是 12/12，那件事你要在 4-1 診斷。

In [ ]:
RUN_C = run_all("C 無評估器", use_evaluator=False)
RUN_C[["id", "型", "行為", "期望行為", "行為正確", "有附條號", "引用查核"]]

# 預期輸出：12 題全部「回答」——沒有評估器就沒有拒答，也沒有反問。
# 拿它跟 A 比，就知道評估器到底換到了什麼、又賠掉了什麼。

### 4-1　系統提示詞與條號

把你的 `EVAL_PROMPT`（TODO 1）**原文**貼在下面。
**貼完整的，不要節錄也不要改寫**——我會照這份原文看你的判準是怎麼寫的。

```
（貼在這裡）
```

**你為什麼這樣寫？** 特別是三個標籤的界線，你怎麼跟模型解釋？

### 條號診斷

`GEN_PROMPT` 明確要求模型附上條號，但它**不會每次都聽**。看你的「有附條號」欄位：

> 最後一問是這一格最有價值的地方。引用查核只驗「這個條號存在於證據裡」，
> **驗不了「條文內容是否真的支持這個回答」**。
> 找到一個「查核通過但答案錯」的例子，你就摸到這個機制的邊界了。

> **請把答案寫在作答本第 9 題**，不要寫在這一格。

### 4-2　有評估器 vs 沒有評估器

跑完 A 與 C 之後填。

### 逐題比對：評估器修好了什麼、又弄壞了什麼

**「多對幾題」這個淨值會騙人。** 把兩次跑的正確題號列出來，分成四類：

### 對照教師基準線

老師用**同樣的架構、同樣的提示詞**跑 Qwen2.5-3B，評估器的三元分類是 **12 題對 8 題**。
而在有產出回答的題目裡，**條號幾乎都會附上**——真正的問題不是「肯不肯附」，
是「**附的對不對**」。那件事你要在 4-1 診斷。

**這個數字不是及格線，也不是上限。** 它只告訴你「一個普通的實作大概到這裡」。

> 最後這一問是這一格的重點。**能分辨「我寫得不好」和「模型就是做不到」，
> 才不會把時間花在修一個不是自己造成的問題上。**

> **請把答案寫在作答本第 10 題**，不要寫在這一格。

In [ ]:
# 重搜的兩種分類（不用改）
#
# 同一次重搜，用兩把不同的尺量：
#   尺一（看片段）：有沒有撈到新的片段？        → 有效 / 無效
#   尺二（看答案）：對回答有沒有幫助？          → 有益 / 中立 / 有害
#
# 這兩把尺**會給出不一樣的答案**，那個落差就是這一格的重點。

rows = []
for _, r in RUN_A.iterrows():
    if not r["重搜"]:
        continue
    item = next(e for e in EVAL if e["id"] == r["id"])
    if not item["答案值"]:
        continue        # 拒答題沒有數值答案，量不出「有沒有新答案」
    before = search(item["q"])
    after = search(r["改寫查詢"])
    ids_b = {(c["file"], c["seq"]) for c in before}
    ids_a = {(c["file"], c["seq"]) for c in after}
    v_b = values_in(before, item)          # ← 只比對評測集標註的答案值
    v_a = values_in(after, item)
    新增 = sorted(v_a - v_b)
    消失 = sorted(v_b - v_a)
    尺一 = "有效" if ids_a - ids_b else "無效"
    尺二 = "有害" if 消失 else ("中立" if not 新增 else "有益")
    rows.append({"題": r["id"], "型": r["型"], "新片段": len(ids_a - ids_b),
                 "值(前)": sorted(v_b), "值(後)": sorted(v_a),
                 "新增值": 新增, "消失值": 消失,
                 "尺一(片段)": 尺一, "尺二(答案)": 尺二})

RESEARCH = pd.DataFrame(rows)
不一致 = [r["題"] for _, r in RESEARCH.iterrows()
          if r["尺一(片段)"] == "有效" and r["尺二(答案)"] != "有益"]
print("兩把尺不一致的題目：", 不一致 or "無")
print("（尺一說「撈到新片段了」，尺二說「但沒帶來新答案」——那個落差就是重點）")
RESEARCH

# 預期輸出：有觸發重搜、且有標註答案值的題目各一列。
# 你多半會看到「尺一有效、尺二中立」的組合——撈到新片段，卻沒有新答案。

### 4-3　重搜到底值不值得

看上一格的表格填。

**三個追問：**

> **請把答案寫在作答本第 11 題**，不要寫在這一格。

In [ ]:
# 成本統計（不用改）
def cost(df):
    c = df.attrs["calls"]
    by = {}
    for x in c:
        k = x["kind"]
        b = by.setdefault(k, {"次": 0, "in": 0, "out": 0, "秒": 0.0})
        b["次"] += 1; b["in"] += x["in"]; b["out"] += x["out"]; b["秒"] += x["sec"]
    print(f"── {df.attrs['name']} ──")
    for k, b in by.items():
        print(f"  {k:<6} {b['次']:>3} 次　輸入 {b['in']:>7} tok　"
              f"輸出 {b['out']:>5} tok　{b['秒']:>6.1f} 秒")
    tot = {"次": len(c), "in": sum(x["in"] for x in c),
           "out": sum(x["out"] for x in c), "秒": sum(x["sec"] for x in c)}
    print(f"  {'合計':<6} {tot['次']:>3} 次　輸入 {tot['in']:>7} tok　"
          f"輸出 {tot['out']:>5} tok　{tot['秒']:>6.1f} 秒")
    return tot


for df in [RUN_A, RUN_C]:      # 做了加分項的話，把你那一次也加進來
    cost(df)

# 預期輸出：每次跑各一組統計，依「評估／生成」分開列出

### 4-4　成本　→　請寫在**作答本第 12 題**

### 4-5　失敗歸因

**逐題**把處理錯的題目分類。分類依據以你在 4-2、4-3 記錄的資料為準。

| 分類 | 什麼情況算這一類 |
|---|---|
| **資訊不在** | 撈回來的片段裡確實沒有答案 |
| **資訊在但沒讀懂** | 片段裡有答案，模型沒用上 |
| **評估器判錯** | 三元分類給錯了類別 |
| **重搜方向錯** | 改寫後的查詢撈回更差的東西 |
| **引用不實** | 引用查核沒通過 |
| **本來就沒答案** | 語料裡確實沒有 |

> **「資訊不在」和「資訊在但沒讀懂」的區分是這一格的評分重點。**
> 前者要改檢索，後者要改提示詞——分不清楚就會把力氣花在錯的地方。

> **請把答案寫在作答本第 13 題**，不要寫在這一格。

---
# 區塊五　結果評鑑

### 5-1　評估器值得嗎　→　請寫在**作答本第 14 題**

### 5-2　預測核對　→　請寫在**作答本第 15 題**

### 5-3　如果只能留一個　→　請寫在**作答本第 16 題**

### 5-4　回望　→　請寫在**作答本第 17 題**

---
# 加分項（上限 5%，不做不扣分）

## 換一個後端重跑一次

前面兩次跑都用 Colab 上的 Qwen2.5-3B。如果你有其他可用的模型，
換過去重跑一次「有評估器」，看看差在哪裡。

**做法**：回到「後端設定」那一格，把 `BACKEND` 改成你要用的，重新執行那一格，
然後跑下面這一格。

**沒有其他後端可用的話跳過這一項，不影響正課成績。**

> ⚠ **先確認額度夠**。跑一輪 12 題需要**約 26 次呼叫**。
> 有些免費方案的每日上限低於這個數字，跑到一半會中斷。

In [ ]:
# 加分項：換後端重跑（沒做就跳過這一格）
RUN_X = run_all("加分：換後端")
RUN_X[["id", "型", "標籤1", "重搜", "標籤2", "行為", "期望行為", "行為正確",
       "有附條號", "引用查核"]]

### 加分　換後端的差異　→　請寫在**作答本第 18 題**

---
# 繳交前檢查

- [ ] 從頭到尾「重新啟動並全部執行」一次，沒有出錯
- [ ] 四個 TODO 都完成了
- [ ] 兩次跑（A 有評估器／C 無評估器）都有結果
- [ ] 所有 ✍️ 格都有內容
- [ ] **憑證沒有出現在任何一格程式碼裡**
- [ ] 檔名改為 `HW2_學號_姓名.ipynb`

## 我改作業時會看哪幾個地方

只交兩件，但我不是只看答對幾題。下面五件事會逐項看，
先列出來，讓你知道力氣該放在哪裡：

| 我會看 | 在哪裡看 |
|---|---|
| 你讓助理做什麼、明確不讓它做什麼 | 作答本第 3 題與第 7 題 |
| 你最後用的那份系統提示詞原文 | 作答本第 9 題 |
| 一次成功的完整執行紀錄 | **這份 notebook 的輸出**——區塊四用 `run_agent(..., verbose=True)` 跑，紀錄會留在格子的輸出區 |
| 一次失敗的執行紀錄，以及你怎麼診斷、改了什麼 | 作答本第 13 題與第 15 題 |
| 後端、型號、呼叫次數、token、耗時 | 作答本第 12 題 |

> ⚠ 第三項**不用另外附檔**，它就在你繳交的 notebook 裡。
> 「重新啟動並全部執行」跑完**直接存檔，不要清除輸出**，否則這一項等於沒交。
>
> 做了加分項的話，最後一項（作答本第 12 題）要把兩個後端都列出來。